<div dir="rtl" align="right">

# تشخیص ناهنجاری

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
**ناهنجاری** (Anomaly یا Outlier) نمونه‌ای است که با بقیه فرق دارد: تراکنش تقلبی، دستگاه معیوب، حمله شبکه. معمولاً برچسب نداریم و ناهنجاری‌ها خیلی کم‌اند؛ پس مسئله بدون ناظر است.

## روش‌ها
| روش | ایده | sklearn |
|---|---|---|
| آماری (Z-score، IQR) | فاصله از میانگین یا چارک‌ها | دستی |
| Isolation Forest | نقطه عجیب با برش‌های تصادفی **زود جدا** می‌شود | `IsolationForest` |
| LOF | چگالی اطراف نقطه در مقایسه با همسایه‌ها | `LocalOutlierFactor` |
| مبتنی بر مدل | احتمال کم در GMM | `GaussianMixture.score_samples` |
| One-Class SVM | مرزی دور داده عادی | `OneClassSVM` |

## روش IQR (ساده و پرکاربرد)
چارک اول $Q_1$ و سوم $Q_3$؛ $IQR=Q_3-Q_1$. نقاط بیرون از $[Q_1-1.5IQR,\ Q_3+1.5IQR]$ ناهنجارند (همان نقاط نمودار جعبه‌ای).

| داده | $Q_1$ | $Q_3$ | IQR | بازه مجاز | ناهنجار |
|---|---|---|---|---|---|
| 10، 12، 12، 13، 14، 15، 40 | 12 | 14.5 | 2.5 | [8.25, 18.25] | 40 |



</div>

In [ ]:
import numpy as np

x = np.array([10, 12, 12, 13, 14, 15, 40])
q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, allowed=[{low}, {high}]")
print("anomalies:", x[(x < low) | (x > high)])
z = (x - x.mean()) / x.std()
print("z-scores:", z.round(2))

<div dir="rtl" align="right">



## Isolation Forest
درخت‌های تصادفی می‌سازد: ویژگی تصادفی و برش تصادفی. نقطه عادی در وسط جمعیت است و برای جدا شدن **برش‌های زیادی** لازم دارد؛ نقطه ناهنجار با **چند برش** تنها می‌شود. امتیاز ناهنجاری از **میانگین عمق** جداسازی در همه درخت‌ها به دست می‌آید.



</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

rng = np.random.default_rng(0)
normal = rng.normal(0, 1, (300, 2))
odd = rng.uniform(-6, 6, (15, 2))
X = np.r_[normal, odd]

iso = IsolationForest(contamination=0.05, random_state=0).fit(X)
lab_iso = iso.predict(X)                 # -1 = anomaly
lab_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(X)
print("Isolation Forest found", np.sum(lab_iso == -1), "anomalies;", np.sum(lab_iso[300:] == -1), "of the 15 planted ones")
print("LOF found", np.sum(lab_lof == -1), "anomalies;", np.sum(lab_lof[300:] == -1), "of the 15 planted ones")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, lab, t in [(ax[0], lab_iso, "Isolation Forest"), (ax[1], lab_lof, "LOF")]:
    a.scatter(X[:, 0], X[:, 1], c=np.where(lab == -1, "red", "lightgray"), s=12); a.set_title(t)
plt.show()

<div dir="rtl" align="right">



> پارامتر `contamination` سهم ناهنجاری‌ها را مشخص می‌کند و آستانه را تعیین می‌کند. اگر نمی‌دانی، به جای برچسب از `score_samples` استفاده کن و بیشترین امتیازها را بررسی کن.

</div>